# 131 — GNN Behaviour Cloning Shift analysis

Episode `78867640`, focus step **35**.

- Load `130-GNN_BC_shift/model_epoch100.pt`
- `reachable_max_ships` filtered to my planets at step 35
- `actions_to_copy` ground truth (labels use step T, NOT T+1, since 130 shifts internally)
- Model predictions with probabilities

**Key difference from 128:** `130-GNN_BC_shift` was trained with `actions_to_copy[step == T+1]` as labels for observation step T, fixing the assumed off-by-one alignment.

In [1]:
import sys
import importlib.util
import math
import numpy as np
import polars as pl
import pandas as pd
import torch
from pathlib import Path

# Load 130-GNN_BC_shift as a module (name starts with digit, use importlib)
spec = importlib.util.spec_from_file_location('gnn_bc_shift', '130-GNN_BC_shift.py')
gnn_bc = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gnn_bc)
sys.modules['gnn_bc_shift'] = gnn_bc  # torch_geometric Inspector resolves type hints via sys.modules

GNNBehaviourCloning = gnn_bc.GNNBehaviourCloning
build_graph         = gnn_bc.build_graph
load_episode        = gnn_bc.load_episode

EPISODE    = 78867640
FOCUS_STEP = 34
PRE_DIR    = Path(f'126-precompute/{EPISODE}')
MODEL_PATH = Path('130-GNN_BC_shift/model_epoch100.pt')

In [2]:
planete, planete_step, reachable_base_2, reachable_max_ships, actions_to_copy = load_episode(PRE_DIR)

for name, df in [
    ('planete',             planete),
    ('planete_step',        planete_step),
    ('reachable_base_2',    reachable_base_2),
    ('reachable_max_ships', reachable_max_ships),
    ('actions_to_copy',     actions_to_copy),
]:
    print(f'{name:25s}  {str(df.shape):18s}  cols: {df.columns}')

planete                    (17268, 6)          cols: ['id', 'step', 'x', 'y', 'production', 'nature']
planete_step               (344988, 5)         cols: ['id', 'step', 'future_step', 'ships', 'owner']
reachable_base_2           (4172992, 5)        cols: ['id_src', 'step_src', 'id_tgt', 'step_tgt', 'ships_sent']
reachable_max_ships        (284012, 6)         cols: ['id_src', 'step_src', 'angle', 'ships_sent', 'id_tgt', 'step_tgt']
actions_to_copy            (88, 5)             cols: ['step', 'id_src', 'angle', 'ships_sent', 'id_tgt']


In [3]:
planete_step.to_pandas().query("step == @FOCUS_STEP and future_step == @FOCUS_STEP")

,id,step,future_step,ships,owner
30415,19,34,34,9,-1
35319,21,34,34,8,1
53970,31,34,34,28,-1
61975,30,34,34,28,-1
76552,8,34,34,85,-1
85094,22,34,34,8,0
107361,5,34,34,45,-1
145610,29,34,34,28,-1
147871,6,34,34,45,-1
152721,10,34,34,85,-1


## My planets at step 35

In [4]:
T = FOCUS_STEP

ps_now = planete_step.filter(
    (pl.col('step') == T) & (pl.col('future_step') == T)
)
my_planet_ids = ps_now.filter(pl.col('owner') == 0)['id'].to_list()
print(f'My planets at step {T}: {sorted(my_planet_ids)}')

ps_now.sort('id').to_pandas()

My planets at step 34: [12, 18, 20, 22]


,id,step,future_step,ships,owner
0,0,34,34,43,-1
1,1,34,34,43,-1
2,2,34,34,43,-1
3,3,34,34,43,-1
4,4,34,34,45,-1
5,5,34,34,45,-1
6,6,34,34,45,-1
7,7,34,34,45,-1
8,8,34,34,85,-1
9,9,34,34,85,-1


## `reachable_max_ships` — owned by me, step_src == 35

In [5]:
reachable_mine = reachable_max_ships.filter(
    (pl.col('step_src') == T) & (pl.col('id_src').is_in(my_planet_ids))
).sort(['id_src', 'id_tgt'])

print(f'{len(reachable_mine)} candidate actions from my {len(my_planet_ids)} planet(s)')
reachable_mine.to_pandas()

40 candidate actions from my 4 planet(s)


,id_src,step_src,angle,ships_sent,id_tgt,step_tgt
0,12,34,2.174115,1,5,50
1,12,34,0.414277,1,14,54
2,12,34,0.803769,1,18,51
3,12,34,1.270625,1,20,38
4,12,34,2.883870,1,25,51
5,12,34,-0.015297,1,30,54
6,18,34,1.585341,18,0,39
7,18,34,-0.007727,18,4,42
8,18,34,-0.434972,18,8,46
9,18,34,2.854355,18,9,52


## `actions_to_copy` — ground truth

130-GNN_BC_shift trains with `actions_to_copy[step == T+1]` as labels for observation T.
So for the analysis we show both:
- **step T** (what 128 used)
- **step T+1** (what 130 was trained on)

In [6]:
actions_T   = actions_to_copy.filter(pl.col('step') == T)
actions_T1  = actions_to_copy.filter(pl.col('step') == T + 1)

print(f'{len(actions_T)} ground truth action(s) at step {T} (128-style)')
display(actions_T.to_pandas())

print(f'{len(actions_T1)} ground truth action(s) at step {T+1} (130-style, what model was trained on)')
display(actions_T1.to_pandas())

1 ground truth action(s) at step 34 (128-style)


,step,id_src,angle,ships_sent,id_tgt
0,34,12,1.009923,13,20


1 ground truth action(s) at step 35 (130-style, what model was trained on)


,step,id_src,angle,ships_sent,id_tgt
0,35,18,-1.4326,18,14


## Load model — 130-GNN_BC_shift epoch 100

In [7]:
model = GNNBehaviourCloning(
    H  = gnn_bc.HIDDEN_DIM,
    L1 = gnn_bc.NUM_LAYERS_P1,
    L2 = gnn_bc.NUM_LAYERS_P2,
)
state_dict = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
model.load_state_dict(state_dict)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Loaded  : {MODEL_PATH}')
print(f'Params  : {n_params:,}')

Loaded  : 130-GNN_BC_shift\model_epoch100.pt
Params  : 142,785


## Build graph for step 35 and run inference

Note: `build_graph` in 130 uses `actions_to_copy[step == T+1]` internally for labels.

In [8]:
graph = build_graph(
    T,
    planete, planete_step, reachable_base_2, reachable_max_ships, actions_to_copy
)
assert graph is not None, 'build_graph returned None — no action_max nodes at this step'

n_ams = graph['action_max'].x.shape[0]
n_pos = int(graph['action_max'].y.sum().item())
print(f'planet nodes       : {graph["planet"].x.shape[0]}')
print(f'planet_step nodes  : {graph["planet_step"].x.shape[0]}')
print(f'action_max nodes   : {n_ams}  (positives={n_pos})')
print()
print(graph)

planet nodes       : 32
planet_step nodes  : 672
action_max nodes   : 40  (positives=1)

HeteroData(
  planet={ x=[32, 4] },
  planet_step={ x=[672, 9] },
  action_max={
    x=[40, 1],
    y=[40],
  },
  (planet, has_ps, planet_step)={ edge_index=[2, 672] },
  (planet_step, rev_has_ps, planet)={ edge_index=[2, 672] },
  (planet_step, reaches, planet_step)={
    edge_index=[2, 23865],
    edge_attr=[23865],
  },
  (planet_step, src_of, action_max)={ edge_index=[2, 40] },
  (action_max, rev_src_of, planet_step)={ edge_index=[2, 40] },
  (planet_step, tgt_of, action_max)={ edge_index=[2, 40] },
  (action_max, rev_tgt_of, planet_step)={ edge_index=[2, 40] }
)


In [9]:
with torch.no_grad():
    logits = model(graph)          # (n_ams,)
    probs  = torch.sigmoid(logits).numpy()

print(f'logits : [{logits.min():.3f}, {logits.max():.3f}]')
print(f'probs  : [{probs.min():.3f}, {probs.max():.3f}]')

logits : [-4.530, 3.448]
probs  : [0.011, 0.969]


## Predicted actions vs ground truth

Reconstruct which `(id_src, id_tgt)` corresponds to each `action_max` node
by replicating the validity filter from `build_graph`.

Labels here reflect `actions_to_copy[step == T+1]` (130-style).

In [10]:
# ── Rebuild PS key index ──────────────────────────────────────────────────────
ps_raw          = planete_step.filter(pl.col('step') == T)
ps_ids_np       = ps_raw['id'].to_numpy()
ps_fstep_np     = ps_raw['future_step'].to_numpy()
ps_key_to_idx   = {
    (int(ps_ids_np[i]), int(ps_fstep_np[i])): i
    for i in range(len(ps_raw))
}

# ── ams_labeled (same join as in build_graph — uses T+1 labels) ──────────────
ams_raw = reachable_max_ships.filter(
    (pl.col('step_src') == T) & (pl.col('id_src').is_in(my_planet_ids))
)
act_flag = (
    actions_to_copy
    .filter(pl.col('step') == T + 1)   # 130-style shift
    .select(['id_src', 'id_tgt'])
    .with_columns(pl.lit(1).alias('_match'))
)
ams_labeled = (
    ams_raw
    .join(act_flag, on=['id_src', 'id_tgt'], how='left')
    .with_columns(label=pl.col('_match').fill_null(0).cast(pl.Int64))
    .drop('_match')
)

# ── Validity filter: same as build_graph valid_ams_rows ───────────────────────
id_src_np   = ams_labeled['id_src'].to_numpy()
id_tgt_np   = ams_labeled['id_tgt'].to_numpy()
step_tgt_np = ams_labeled['step_tgt'].to_numpy()

valid_rows = [
    i for i in range(len(ams_labeled))
    if ps_key_to_idx.get((int(id_src_np[i]), T), -1) >= 0
    and ps_key_to_idx.get((int(id_tgt_np[i]), int(step_tgt_np[i])), -1) >= 0
]

assert len(valid_rows) == n_ams, (
    f'Mismatch: valid_rows={len(valid_rows)}, action_max nodes={n_ams}'
)

# ── Attach predictions ────────────────────────────────────────────────────────
result = (
    ams_labeled[valid_rows]
    .with_columns([
        pl.Series('prob',      probs.tolist()),
        pl.Series('predicted', (probs >= 0.5).tolist()),
    ])
    .sort(['id_src', 'id_tgt'])
)

print(f'action_max nodes: {n_ams}  positives (T+1 labels): {n_pos}  predicted positives: {int((probs>=0.5).sum())}')
result.to_pandas()

action_max nodes: 40  positives (T+1 labels): 1  predicted positives: 11


,id_src,step_src,angle,ships_sent,id_tgt,step_tgt,label,prob,predicted
0,12,34,2.174115,1,5,50,0,0.548847,True
1,12,34,0.414277,1,14,54,0,0.043928,False
2,12,34,0.803769,1,18,51,0,0.131581,False
3,12,34,1.270625,1,20,38,0,0.775344,True
4,12,34,2.883870,1,25,51,0,0.210374,False
5,12,34,-0.015297,1,30,54,0,0.037911,False
6,18,34,1.585341,18,0,39,0,0.151486,False
7,18,34,-0.007727,18,4,42,0,0.924169,True
8,18,34,-0.434972,18,8,46,0,0.025447,False
9,18,34,2.854355,18,9,52,0,0.014762,False


## Positives only — ground truth (T+1) and predicted side by side

In [11]:
pos_gt   = result.filter(pl.col('label') == 1)
pos_pred = result.filter(pl.col('predicted') == True)

print('=== Ground truth positive actions (step T+1 labels) ===')
display(pos_gt.select(['id_src','id_tgt','ships_sent','step_tgt','prob','label']).to_pandas())

print('=== Predicted positive actions (prob >= 0.5) ===')
display(pos_pred.select(['id_src','id_tgt','ships_sent','step_tgt','prob','label']).to_pandas())

# ── top-k by prob ─────────────────────────────────────────────────────────────
print('=== Top 10 by probability ===')
display(
    result
    .sort('prob', descending=True)
    .head(10)
    .select(['id_src','id_tgt','ships_sent','step_tgt','prob','label','predicted'])
    .to_pandas()
)

=== Ground truth positive actions (step T+1 labels) ===


,id_src,id_tgt,ships_sent,step_tgt,prob,label
0,18,14,18,37,0.957426,1


=== Predicted positive actions (prob >= 0.5) ===


,id_src,id_tgt,ships_sent,step_tgt,prob,label
0,12,5,1,50,0.548847,0
1,12,20,1,38,0.775344,0
2,18,4,18,42,0.924169,0
3,18,14,18,37,0.957426,1
4,18,22,18,39,0.961009,0
5,18,24,18,38,0.880992,0
6,20,5,27,40,0.923396,0
7,20,12,27,41,0.969186,0
8,20,18,27,42,0.899902,0
9,22,4,8,43,0.857408,0


=== Top 10 by probability ===


,id_src,id_tgt,ships_sent,step_tgt,prob,label,predicted
0,20,12,27,41,0.969186,0,True
1,18,22,18,39,0.961009,0,True
2,18,14,18,37,0.957426,1,True
3,18,4,18,42,0.924169,0,True
4,20,5,27,40,0.923396,0,True
5,20,18,27,42,0.899902,0,True
6,18,24,18,38,0.880992,0,True
7,22,4,8,43,0.857408,0,True
8,12,20,1,38,0.775344,0,True
9,22,19,8,44,0.749215,0,True


## Comparison: 128 labels (step T) vs 130 labels (step T+1)

Shows which candidate actions flip between the two label conventions.

In [12]:
# Rebuild with T-labels for comparison
act_flag_128 = (
    actions_to_copy
    .filter(pl.col('step') == T)
    .select(['id_src', 'id_tgt'])
    .with_columns(pl.lit(1).alias('label_128'))
)
result_cmp = (
    result
    .join(act_flag_128, on=['id_src', 'id_tgt'], how='left')
    .with_columns(pl.col('label_128').fill_null(0).cast(pl.Int64))
    .rename({'label': 'label_130'})
    .select(['id_src', 'id_tgt', 'ships_sent', 'step_tgt', 'prob', 'predicted', 'label_128', 'label_130'])
    .sort(['id_src', 'id_tgt'])
)

print('Rows where labels differ between 128 and 130:')
display(result_cmp.filter(pl.col('label_128') != pl.col('label_130')).to_pandas())

print('\nFull comparison:')
result_cmp.to_pandas()

Rows where labels differ between 128 and 130:


,id_src,id_tgt,ships_sent,step_tgt,prob,predicted,label_128,label_130
0,12,20,1,38,0.775344,True,1,0
1,18,14,18,37,0.957426,True,0,1



Full comparison:


,id_src,id_tgt,ships_sent,step_tgt,prob,predicted,label_128,label_130
0,12,5,1,50,0.548847,True,0,0
1,12,14,1,54,0.043928,False,0,0
2,12,18,1,51,0.131581,False,0,0
3,12,20,1,38,0.775344,True,1,0
4,12,25,1,51,0.210374,False,0,0
5,12,30,1,54,0.037911,False,0,0
6,18,0,18,39,0.151486,False,0,0
7,18,4,18,42,0.924169,True,0,0
8,18,8,18,46,0.025447,False,0,0
9,18,9,18,52,0.014762,False,0,0
